# ARC-AGI-3 E1 — one-click Qwen3.8 benchmark

Attach the Qwen3.8 model dataset, this `arc3-e1-bundle`, and the existing `ARC3 vLLM H100 Wheelhouse V3`; select an RTX Pro 6000 and use **Run All**. The first cell discovers the bundle/wheelhouse and validates every required mount and runtime package before installation or model startup.

In [ ]:
import importlib.util, json, os, subprocess, sys, time
from pathlib import Path

# Locked E0/E1 target. This notebook intentionally does not read a historical model alias.
MODEL_DATASET = 'foysalemonshanto/qwen3-8-27b-fp8-repacked-v1'
MODEL_DATASET_SLUG = MODEL_DATASET.rsplit('/', 1)[-1]
MODEL_ID = None  # resolved below from the attached Qwen3.8 dataset
INPUT_ROOT = Path('/kaggle/input')
WHEELHOUSE_DATASET_SLUG = 'arc3-vllm-h100-wheelhouse-v3'
KNOWN_WHEELHOUSE = INPUT_ROOT / WHEELHOUSE_DATASET_SLUG
WORKDIR = Path('/kaggle/working')
BASE_URL = 'http://127.0.0.1:1234/v1'
SERVED_MODEL_NAME = None

def _bundle_complete(path):
    required = (path / 'arc3', path / 'kaggle/e1_qwen/serve_qwen.sh',
                 path / 'kaggle/e1_qwen/e1_runner.py',
                 path / 'kaggle/e1_qwen/model_metadata.py',
                 path / 'artifacts/public_games.json')
    return path.is_dir() and all(item.exists() for item in required)

def _bundle_manifest_paths(search_root):
    if not search_root.is_dir():
        return []
    # Kaggle can expose datasets directly, under datasets/<owner>/<slug>,
    # or with an extra directory layer. Search all mounted descendants.
    return sorted((item for item in search_root.rglob('bundle_manifest.json') if item.is_file()), key=str)

def _valid_bundle_candidates(search_root):
    valid = {}
    manifests = _bundle_manifest_paths(search_root)
    malformed = []
    for manifest_path in manifests:
        try:
            manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
        except (OSError, UnicodeDecodeError, json.JSONDecodeError) as exc:
            malformed.append(f'{manifest_path}: {exc}')
            continue
        if not isinstance(manifest, dict) or manifest.get('bundle') != 'arc3-e1-bundle':
            continue
        root = manifest_path.parent.resolve()
        if _bundle_complete(root):
            valid[root] = manifest_path.resolve()
    return valid, manifests, malformed

def _discover_bundle():
    explicit = os.environ.get('E1_BUNDLE_DIR', '').strip()
    search_root = Path(explicit).expanduser() if explicit else INPUT_ROOT
    valid, manifests, malformed = _valid_bundle_candidates(search_root)
    if len(valid) > 1:
        details = '\n  - '.join(f'{root} (manifest: {manifest})' for root, manifest in sorted(valid.items(), key=lambda pair: str(pair[0])))
        raise RuntimeError('Ambiguous arc3-e1-bundle: multiple complete manifests were found under ' + str(search_root) + ':\n  - ' + details + '\nRemove the duplicate mount or set E1_BUNDLE_DIR explicitly.')
    if len(valid) == 1:
        return next(iter(valid))
    if not manifests:
        label = 'explicit E1_BUNDLE_DIR' if explicit else 'recursive Kaggle input root'
        raise RuntimeError(f'bundle_manifest.json not found under {label}: {search_root}')
    if malformed:
        details = '\n  - '.join(malformed)
        raise RuntimeError('Found bundle_manifest.json files, but none was a valid complete arc3-e1-bundle. Parse errors:\n  - ' + details)
    found = '\n  - '.join(str(item.resolve()) for item in manifests)
    raise RuntimeError('Found bundle_manifest.json files, but none described a complete arc3-e1-bundle:\n  - ' + found)

def _model_complete(path):
    if not path.is_dir() or not (path / 'config.json').is_file():
        return False
    try:
        config = json.loads((path / 'config.json').read_text(encoding='utf-8'))
    except (OSError, UnicodeDecodeError, json.JSONDecodeError):
        return False
    if not isinstance(config, dict):
        return False
    # A HF checkpoint must contain at least one local weight/index marker.
    weight_markers = ('*.safetensors', '*.bin', '*.pt', '*.pth', '*.gguf')
    return any(any(path.glob(pattern)) for pattern in weight_markers)

def _path_has_slug(path, slug):
    needle = slug.lower().replace('_', '-')
    return any(needle in part.lower().replace('_', '-') for part in path.parts)

def _discover_model():
    explicit = os.environ.get('E1_MODEL_ID', '').strip()
    if explicit:
        path = Path(explicit).expanduser().resolve()
        if not _path_has_slug(path, MODEL_DATASET_SLUG):
            raise RuntimeError('E1_MODEL_ID points outside the locked Qwen3.8-27B-FP8 Repacked dataset; Qwen3.6 fallback is disabled: ' + str(path))
        if not _model_complete(path):
            raise RuntimeError('Explicit E1_MODEL_ID is not a complete local HF checkpoint: ' + str(path))
        return path
    candidates = []
    if INPUT_ROOT.is_dir():
        for config_path in INPUT_ROOT.rglob('config.json'):
            root = config_path.parent.resolve()
            if _path_has_slug(root, MODEL_DATASET_SLUG) and _model_complete(root):
                candidates.append(root)
    candidates = sorted(set(candidates), key=str)
    if len(candidates) > 1:
        details = '\n  - '.join(str(item) for item in candidates)
        raise RuntimeError('Ambiguous locked Qwen3.8 model: multiple complete checkpoints were found under /kaggle/input:\n  - ' + details + '\nRemove the duplicate mount or set E1_MODEL_ID explicitly.')
    if len(candidates) == 1:
        return candidates[0]
    raise RuntimeError('Locked Qwen3.8-27B-FP8 Repacked model checkpoint not found recursively under /kaggle/input; attach the model dataset ' + MODEL_DATASET)

def _wheel_files(path):
    return sorted(path.rglob('*.whl')) if path is not None and path.is_dir() else []

def _discover_wheelhouse():
    # E0 used ARC3 vLLM H100 Wheelhouse V3. Kaggle can mount it under
    # /datasets/<owner>/<slug>/..., and E1 may also need a small separate
    # pure-Python toolkit wheel input. Combine recognized roots; fail on
    # duplicate primary/toolkit inputs instead of choosing arbitrarily.
    explicit = os.environ.get('E1_WHEEL_DIR', '').strip()
    if explicit:
        paths = [Path(item).expanduser().resolve() for item in explicit.split(os.pathsep) if item.strip()]
        roots = [(path, _wheel_files(path)) for path in paths]
        roots = [(path, files) for path, files in roots if files]
        if not roots:
            raise RuntimeError('Explicit E1_WHEEL_DIR contains no .whl files: ' + explicit)
        return os.pathsep.join(str(path) for path, _ in roots), sorted({item for _, files in roots for item in files}, key=str)
    if not INPUT_ROOT.is_dir():
        return None, []
    wheels = sorted((item for item in INPUT_ROOT.rglob('*.whl') if item.is_file()), key=str)
    roots = {}
    slug_key = WHEELHOUSE_DATASET_SLUG.lower().replace('_', '-')
    toolkit_prefixes = ('arc-agi-', 'arcengine-')
    for wheel in wheels:
        filename = wheel.name.lower().replace('_', '-')
        is_toolkit = filename.startswith(toolkit_prefixes)
        ancestors = [wheel.parent.resolve(), *[parent.resolve() for parent in wheel.parents]]
        root = next((parent for parent in ancestors if slug_key in parent.name.lower().replace('_', '-')), None)
        if root is None and is_toolkit:
            root = next((parent for parent in ancestors if 'wheelhouse' in parent.name.lower() or 'vllm' in parent.name.lower() or 'wheels' in parent.name.lower()), None)
        if root is None:
            root = next((parent for parent in ancestors if 'wheelhouse' in parent.name.lower() or 'vllm' in parent.name.lower()), None)
        if root is not None:
            roots.setdefault(root, []).append(wheel)
    candidates = sorted(((root, sorted(set(files), key=str)) for root, files in roots.items()), key=lambda pair: str(pair[0]))
    if not candidates:
        return None, []
    primary = [(root, files) for root, files in candidates if slug_key in root.name.lower().replace('_', '-') or any(item.name.lower().replace('_', '-').startswith('vllm-') for item in files)]
    toolkit = [(root, files) for root, files in candidates if any(item.name.lower().replace('_', '-').startswith(toolkit_prefixes) for item in files)]
    unknown = [(root, files) for root, files in candidates if root not in {item[0] for item in primary + toolkit}]
    if len(primary) > 1 or len(toolkit) > 1 or unknown:
        details = '\n  - '.join(f'{root} ({len(files)} wheels)' for root, files in candidates)
        raise RuntimeError('Ambiguous ARC3 wheel inputs were found under /kaggle/input:\n  - ' + details + '\nKeep one primary vLLM wheelhouse and at most one toolkit wheel input, or set E1_WHEEL_DIR explicitly.')
    selected = primary + [item for item in toolkit if item[0] not in {root for root, _ in primary}]
    return os.pathsep.join(str(root) for root, _ in selected), sorted({item for _, files in selected for item in files}, key=str)

BUNDLE = _discover_bundle()
BUNDLE_MANIFEST = BUNDLE / 'bundle_manifest.json'
MODEL_ID = _discover_model()
SERVED_MODEL_NAME = str(MODEL_ID)
WHEEL_DIR, wheel_files = _discover_wheelhouse()
required_dirs = {'model': Path(MODEL_ID), 'bundle': BUNDLE}
missing = [f'{name}: {path}' for name, path in required_dirs.items() if not path.is_dir()]
required_files = [Path(MODEL_ID) / 'config.json', BUNDLE / 'arc3', BUNDLE / 'kaggle/e1_qwen/serve_qwen.sh', BUNDLE / 'kaggle/e1_qwen/e1_runner.py', BUNDLE / 'kaggle/e1_qwen/model_metadata.py', BUNDLE / 'artifacts/public_games.json']
missing += [f'file: {path}' for path in required_files if not path.exists()]

# The wheelhouse is optional only when the Kaggle image already provides
# every required top-level import. Otherwise fail before pip/server/actions.
def _discover_source_module(module_name):
    # Competition/dataset mounts may carry pure-Python toolkit sources even
    # when the package is not installed in the Kaggle image.
    roots = sorted({item.parent.parent.resolve() for item in INPUT_ROOT.rglob('__init__.py')
                    if item.parent.name == module_name}, key=str)
    if len(roots) > 1:
        details = '\n  - '.join(str(root) for root in roots)
        raise RuntimeError('Ambiguous source package ' + module_name + ' under /kaggle/input:\n  - ' + details)
    if roots and str(roots[0]) not in sys.path:
        sys.path.insert(0, str(roots[0]))
    return roots[0] if roots else None

sys.path.insert(0, str(BUNDLE))
_discover_source_module('arc_agi')
_discover_source_module('arcengine')
required_modules = {'vllm': 'vllm', 'arc-agi': 'arc_agi', 'arcengine': 'arcengine'}
installed_modules = {dist: importlib.util.find_spec(module) is not None for dist, module in required_modules.items()}
WHEEL_ALIASES = {'arc-agi': ('arc-agi', 'arc_agi'), 'arcengine': ('arcengine',),
                 'vllm': ('vllm',), 'torch': ('torch',), 'transformers': ('transformers',),
                 'tokenizers': ('tokenizers',), 'fastapi': ('fastapi',), 'uvicorn': ('uvicorn',),
                 'pandas': ('pandas',), 'pyarrow': ('pyarrow',)}
def _has_wheel_for(dist):
    aliases = WHEEL_ALIASES.get(dist, (dist,))
    return any(any(Path(item).name.lower().replace('_', '-').startswith(alias + '-') for alias in aliases) for item in wheel_files)
missing_packages = [dist for dist, present in installed_modules.items() if not present and not _has_wheel_for(dist)]
artifact_modules = {'pandas': 'pandas', 'pyarrow': 'pyarrow'}
artifact_status = {dist: importlib.util.find_spec(module) is not None for dist, module in artifact_modules.items()}
if not any(artifact_status.values()) and not any(_has_wheel_for(dist) for dist in artifact_modules):
    missing.append('one parquet writer is required (preinstalled pandas/pyarrow or a matching wheel)')
if missing_packages:
    location = str(WHEEL_DIR) if WHEEL_DIR else 'ARC3 vLLM H100 Wheelhouse V3 (not found)'
    missing.append('required packages unavailable (preinstalled or wheelhouse): ' + ', '.join(missing_packages))
    missing.append('expected wheelhouse mount: ' + str(KNOWN_WHEELHOUSE) + '; discovered: ' + location)
if missing:
    raise RuntimeError('Missing required Kaggle Inputs or bundle files; attach/fix these before Run All:\n  - ' + '\n  - '.join(missing))

# Set the complete preflight configuration automatically.
defaults = {
    'E1_MODEL_ID': MODEL_ID, 'E1_SERVED_MODEL_NAME': SERVED_MODEL_NAME,
    'E1_BUNDLE_DIR': str(BUNDLE), 'E1_WHEEL_DIR': str(WHEEL_DIR) if WHEEL_DIR else '',
    'E1_BASE_URL': BASE_URL, 'E1_METADATA_PATH': str(WORKDIR / 'e1_model_metadata.json'),
    'E1_TOOLKIT_MODE': 'OFFLINE', 'E1_GAME_SET': 'small',
    'E1_MAX_ACTIONS': '30', 'E1_MAX_MODEL_LEN': '32768',
    'E1_TEMPERATURE': '0.6', 'E1_TOP_P': '0.95', 'E1_TOP_K': '20', 'E1_MAX_TOKENS': '2048',
    'E1_ENABLE_THINKING': 'true', 'E1_PRESERVE_THINKING': 'true',
    'E1_REASONING_PARSER': 'qwen3', 'E1_TOOL_CALL_PARSER': 'qwen3_coder',
    'E1_SERVER_WAIT_SECONDS': '900',
    'HF_HUB_OFFLINE': '1', 'TRANSFORMERS_OFFLINE': '1', 'HF_DATASETS_OFFLINE': '1',
    'PIP_NO_INDEX': '1', 'PIP_DISABLE_PIP_VERSION_CHECK': '1',
}
os.environ.update(defaults)
if WORKDIR.is_dir():
    os.chdir(WORKDIR)
print(f'RESOLVED_E1_BUNDLE_DIR={BUNDLE}')
print(f'RESOLVED_E1_MODEL_DIR={MODEL_ID}')
print(f'RESOLVED_E1_WHEELHOUSE_DIR={WHEEL_DIR if WHEEL_DIR else "PREINSTALLED"}')
print(f'RESOLVED_E1_WHEEL_COUNT={len(wheel_files)}')
preflight = {'status': 'preflight_ok', 'model_dataset': MODEL_DATASET, 'model_path': str(MODEL_ID), 'bundle': str(BUNDLE), 'bundle_manifest': str(BUNDLE_MANIFEST), 'wheelhouse_dataset': 'ARC3 vLLM H100 Wheelhouse V3', 'wheelhouse': str(WHEEL_DIR) if WHEEL_DIR else 'PREINSTALLED', 'wheel_count': len(wheel_files), 'installed_modules_before_pip': installed_modules, 'artifact_writer_status_before_pip': artifact_status, 'wheel_inventory': {dist: _has_wheel_for(dist) for dist in WHEEL_ALIASES}, 'benchmark': ['cd82-fb555c5d', 'ls20-9607627b', 'lf52-271a04aa'], 'gpu_target': 'RTX Pro 6000', 'internet': 'OFF'}
(WORKDIR / 'e1_preflight.json').write_text(json.dumps(preflight, indent=2), encoding='utf-8')
print(preflight)
print('E1_PREFLIGHT_OK')

## Install only attached wheels

The next cell uses `--no-index`; it cannot download packages from the network. It installs only missing top-level packages from the discovered E0 wheelhouse; `arc-agi` and `arcengine` are reused from the image when already importable.

In [ ]:
missing_targets = [dist for dist, module in required_modules.items() if not installed_modules[dist]]
if not any(artifact_status.values()):
    missing_targets.append(next((dist for dist in artifact_modules if _has_wheel_for(dist)), ''))
missing_targets = [item for item in missing_targets if item]
if missing_targets:
    if not wheel_files or WHEEL_DIR is None:
        raise RuntimeError('Required packages are not preinstalled and no E0 wheelhouse was discovered: ' + ', '.join(missing_targets))
    link_args = []
    for parent in sorted({item.parent for item in wheel_files}, key=str):
        link_args.extend(['--find-links', str(parent)])
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--no-index', '--disable-pip-version-check', '--upgrade-strategy', 'only-if-needed', *link_args, *missing_targets])
else:
    print('offline wheel install skipped: vLLM, arc-agi, and arcengine are already importable')
post_install = {dist: importlib.util.find_spec(module) is not None for dist, module in required_modules.items()}
post_artifact = {dist: importlib.util.find_spec(module) is not None for dist, module in artifact_modules.items()}
not_importable = [dist for dist, present in post_install.items() if not present]
if not any(post_artifact.values()):
    not_importable.append('pandas-or-pyarrow')
if not_importable:
    raise RuntimeError('Offline install did not provide required imports: ' + ', '.join(not_importable))
print({'offline_wheelhouse': str(WHEEL_DIR) if WHEEL_DIR else 'PREINSTALLED', 'wheel_count': len(wheel_files), 'installed_modules_after_pip': post_install, 'artifact_writer_after_pip': post_artifact})

## Start vLLM and verify localhost health

The launcher prints and saves checkpoint/config metadata before loading weights. The runner then verifies `/v1/models` before any environment action is spent.

In [ ]:
server_script = BUNDLE / 'kaggle/e1_qwen/serve_qwen.sh'
server_log_path = WORKDIR / 'e1_vllm.log'
server_log = server_log_path.open('w', encoding='utf-8')
server = subprocess.Popen(['bash', str(server_script)], env=dict(os.environ), stdout=server_log, stderr=subprocess.STDOUT)
print('vLLM pid:', server.pid, 'log:', server_log_path)
import importlib.util
runner_path = BUNDLE / 'kaggle/e1_qwen/e1_runner.py'
runner_spec = importlib.util.spec_from_file_location('e1_runner', runner_path)
runner_module = importlib.util.module_from_spec(runner_spec)
runner_spec.loader.exec_module(runner_module)
runner_module.wait_for_server(BASE_URL, int(os.environ['E1_SERVER_WAIT_SECONDS']))
print('vLLM health check passed:', BASE_URL)

## Run the fixed three-game E1 benchmark

This keeps the existing `ReasoningAgent` unchanged and runs only the fixed subset with seed 0 and 30 actions per game.

In [ ]:
run_e1 = runner_module.main
run_e1()
result_path = WORKDIR / 'e1_results.json'
metadata_path = WORKDIR / 'e1_model_metadata.json'
if not result_path.exists():
    raise RuntimeError(f'E1 runner completed without {result_path}')
summary = json.loads(result_path.read_text(encoding='utf-8'))
print({k: summary.get(k) for k in ('overall_score', 'levels_cleared', 'actions', 'no_op_rate', 'model_path', 'served_model_name', 'sampling', 'thinking', 'max_model_len')})
print('metadata:', metadata_path if metadata_path.exists() else 'MISSING')
print('artifacts:', {'results': str(result_path), 'metadata': str(metadata_path), 'traces': str(WORKDIR / 'e1_traces'), 'vllm_log': str(server_log_path), 'parquet': str(WORKDIR / 'submission.parquet')})